# ✅ EDA Worksheet — Solution Notebook
### Auto-MPG Dataset

This notebook contains complete, correct solutions to every Task and Question in the worksheet.


## Task 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', None)
print("Libraries loaded ✓")


## Task 2 — Load & Inspect Data

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"
col_names = ['mpg','cylinders','displacement','horsepower','weight',
             'acceleration','model_year','origin','car_name']

df = pd.read_csv(url, names=col_names, sep=r'\s+', na_values='?')
print("Shape:", df.shape)
df.head()


In [ ]:
df.info()

In [ ]:
df.describe()

**A1.** The dataset has **398 rows** and **9 columns**.  
**A2.** The `horsepower` column has **6** missing values (≈1.5% of data).


## Task 3 — Pre-Processing

In [ ]:
# Missing values
print("Missing values:\n", df.isnull().sum())
df.dropna(inplace=True)
print("\nShape after dropping NaN rows:", df.shape)


In [ ]:
# Duplicates
print("Duplicate rows:", df.duplicated().sum())
df.drop_duplicates(inplace=True)


In [ ]:
# Type fixes
df['cylinders']  = df['cylinders'].astype('category')
df['model_year'] = df['model_year'].astype('category')
df['origin']     = df['origin'].map({1:'usa', 2:'europe', 3:'japan'})
df['car_name']   = df['car_name'].str.strip().str.lower()
print(df.dtypes)


In [ ]:
cat_cols = ['cylinders', 'origin', 'model_year']
num_cols = ['mpg', 'displacement', 'horsepower', 'weight', 'acceleration']
print("Cat:", cat_cols)
print("Num:", num_cols)


## Task 4 — Feature Engineering

In [ ]:
df['mpg_level'] = pd.cut(df['mpg'],
                          bins=[0, 17, 29, df['mpg'].max()+1],
                          labels=['low','medium','high'],
                          right=False)
print(df['mpg_level'].value_counts())


In [ ]:
df['car_company'] = df['car_name'].str.split().str[0]
print(df['car_company'].value_counts().head(10))


## Task 5 — Categorical EDA

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14,10))
for ax, col in zip(axes.flat, ['origin','cylinders','mpg_level','model_year']):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=ax, palette='Set2')
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel('')
plt.tight_layout(); plt.show()


In [ ]:
for col in ['origin','cylinders','mpg_level']:
    top_val  = df[col].value_counts().index[0]
    top_pct  = df[col].value_counts(normalize=True).iloc[0] * 100
    print(f"{col:12s}: dominant = '{top_val}' → {top_pct:.1f}%")


In [ ]:
ct = pd.crosstab(df['cylinders'], df['origin'])
print(ct)
ct.plot(kind='bar', figsize=(10,5), colormap='Set2')
plt.title('Cylinders by Origin'); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(data=df, x='origin', hue='mpg_level',
              order=['usa','europe','japan'],
              hue_order=['low','medium','high'], palette='Set1')
plt.title('MPG Level by Origin')
plt.tight_layout(); plt.show()


**A3.** **Japan** has no vehicles with low mpg_level.  
**A4.** **4-cylinder** cars are most common at approximately **50.8%** of the dataset.


## Task 6 — Numerical EDA

In [ ]:
fig, axes = plt.subplots(len(num_cols), 2, figsize=(14,18))
for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, ax=axes[i][0], color='steelblue')
    axes[i][0].set_title(f'{col} — Histogram + KDE')
    sns.boxplot(x=df[col], ax=axes[i][1], color='lightcoral')
    axes[i][1].set_title(f'{col} — Boxplot')
plt.tight_layout(); plt.show()


In [ ]:
def tukey_outliers(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    mask = (series < Q1 - 1.5*IQR) | (series > Q3 + 1.5*IQR)
    return series[mask]

print("=== Outlier counts ===")
for col in num_cols:
    n = len(tukey_outliers(df[col]))
    print(f"  {col:15s}: {n}")


In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5, square=True)
plt.title('Correlation Matrix')
plt.tight_layout(); plt.show()

print("\nCorrelations with mpg:")
print(df[num_cols].corr()['mpg'].sort_values())


In [ ]:
sns.pairplot(df[num_cols + ['mpg_level']], hue='mpg_level',
             hue_order=['low','medium','high'],
             plot_kws={'alpha':0.5}, palette='Set1')
plt.suptitle('Pairplot coloured by MPG Level', y=1.01)
plt.show()


**A5.** **Weight** (r ≈ -0.83) and **displacement** (r ≈ -0.80) have the strongest negative correlations with mpg.  
**A6.** Yes — `acceleration` is the only feature whose histogram is roughly bell-shaped and symmetric, making it approximately Gaussian.


## Task 7 — Numerical vs Categorical

In [ ]:
fig, axes = plt.subplots(1, len(num_cols), figsize=(20,5))
for ax, col in zip(axes, num_cols):
    sns.boxenplot(data=df, x='origin', y=col,
                  order=['usa','europe','japan'], palette='Set2', ax=ax)
    ax.set_title(col); ax.set_xlabel('')
plt.suptitle('Numerical Features by Origin')
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(num_cols), figsize=(20,5))
for ax, col in zip(axes, num_cols):
    sns.violinplot(data=df, x='mpg_level', y=col,
                   order=['low','medium','high'], palette='Set1', ax=ax)
    ax.set_title(col); ax.set_xlabel('')
plt.suptitle('Numerical Features by MPG Level')
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(12,5))
df.groupby('model_year')['mpg'].mean().plot(marker='o', color='steelblue')
plt.title('Average MPG by Model Year')
plt.xlabel('Model Year'); plt.ylabel('Mean MPG')
plt.tight_layout(); plt.show()


**A7.** **USA** produces cars with the lowest average mpg.  
**A8.** MPG increases steadily as model year advances, reflecting improved fuel efficiency over time.


## Task 8 — Summary Table (Filled)

| Feature | Relationship with MPG | Strength |
|---|---|---|
| displacement | Negative | Strong (r ≈ -0.80) |
| horsepower | Negative | Strong (r ≈ -0.78) |
| weight | Negative | Strongest (r ≈ -0.83) |
| acceleration | Positive | Weak (r ≈ +0.42) |
| cylinders | Negative (more cylinders → lower mpg) | Moderate–Strong |
| origin | Japan > Europe > USA for mpg | Moderate |
| model_year | Positive (newer = higher mpg) | Moderate–Strong |
